# Proyecto Machine Learning: FEATURE ENGINEERING

### Análisis predictivo de la gestión de la ayuda humanitaria y el impacto de desastres a nivel global

In [2]:
# Librerías

import os
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
base_dir = os.path.dirname(os.path.abspath("__file__"))
csv_path = os.path.join(base_dir, "..", "data", "desastres_y_respuestas_limpio.csv")
df = pd.read_csv(csv_path)

## Tranformaciones logarítmicas (reducir el skewness y mejorar la distribución de las variables)

In [4]:
# Primero copiamos el dataframe limpio para no tocar el original
df_fe = df.copy()

# Variables que necesitan transformación log
cols_log = ['casualties', 'economic_loss', 'aid_amount', 'recovery_days', 'response_hours']

# Creamos columnas nuevas con el sufijo _log — no machacamos las originales
for col in cols_log:
    df_fe[f'{col}_log'] = np.log1p(df_fe[col])

# Comprobamos que la skewness ha bajado
print("Skewness antes y después:")
for col in cols_log:
    antes = df[col].skew()
    despues = df_fe[f'{col}_log'].skew()
    print(f"  {col}: {antes:.2f} → {despues:.2f}")

Skewness antes y después:
  casualties: 157.50 → -0.22
  economic_loss: 8.17 → -0.89
  aid_amount: 7.33 → -0.97
  recovery_days: 3.10 → 0.20
  response_hours: 2.98 → -0.28


In [6]:
df_fe.head()

,year,country,disaster_type,severity_index,casualties,economic_loss,response_hours,aid_amount,response_score,recovery_days,casualties_log,economic_loss_log,aid_amount_log,recovery_days_log,response_hours_log
0,2022,Bangladesh,Flood,0.2,7,1495.0,35.0,368.0,19.51,112.0,2.079442,7.310550,5.910797,4.727388,3.583519
1,2015,Indonesia,Flood,0.1,0,16669.5,2.0,6728.0,100.00,111.0,0.000000,9.721396,8.814182,4.718499,1.098612
2,2023,Indonesia,Landslide,0.6,6,18001.0,10.0,2875.0,80.49,103.0,1.945910,9.798238,7.964156,4.644391,2.397895
3,2016,Bangladesh,Storm Surge,0.2,9,110843.0,47.0,6070.0,0.00,153.0,2.302585,11.615879,8.711279,5.036953,3.871201
4,2018,Japan,Storm Surge,0.6,5,10860.0,8.0,8268.0,85.37,9.0,1.791759,9.292934,9.020269,2.302585,2.197225


#### CUIDADO:
He transformado logarítmicamente uno de mis target (recovery_days). Hay que tenerlo en cuenta para transformalo de nuevo con el siguiente código al hacer las conclusiones:

In [ ]:
# # Transformamos para modelar
# y2 = np.log1p(df_fe['recovery_days'])

# # Al evaluar, deshacemos la transformación
# predicciones_dias_reales = np.expm1(modelo.predict(X2_test))

## Codificación de variables categóricas

In [7]:
# Aplicamos OHE — drop_first=False porque usaremos varios modelos distintos
df_fe = pd.get_dummies(df_fe, columns=['country', 'disaster_type'], drop_first=False)

# Comprobación
print(f"Columnas totales tras OHE: {df_fe.shape[1]}")
print(f"\nColumnas de país:")
print([c for c in df_fe.columns if c.startswith('country_')])
print(f"\nColumnas de tipo de desastre:")
print([c for c in df_fe.columns if c.startswith('disaster_type_')])
print(f"\nTodas las columnas del dataframe:")
print(df_fe.columns.tolist())

Columnas totales tras OHE: 49

Columnas de país:
['country_Australia', 'country_Bangladesh', 'country_Brazil', 'country_Canada', 'country_Chile', 'country_China', 'country_Congo', 'country_France', 'country_Germany', 'country_Greece', 'country_India', 'country_Indonesia', 'country_Ireland', 'country_Italy', 'country_Japan', 'country_Mexico', 'country_New Zealand', 'country_Nigeria', 'country_Peru', 'country_Philippines', 'country_South Africa', 'country_Spain', 'country_Turkey', 'country_United States']

Columnas de tipo de desastre:
['disaster_type_Cyclone', 'disaster_type_Drought', 'disaster_type_Earthquake', 'disaster_type_Extreme Cold', 'disaster_type_Extreme Heat', 'disaster_type_Flood', 'disaster_type_Landslide', 'disaster_type_Storm Surge', 'disaster_type_Tornado', 'disaster_type_Tsunami', 'disaster_type_Volcanic Eruption', 'disaster_type_Wildfire']

Todas las columnas del dataframe:
['year', 'severity_index', 'casualties', 'economic_loss', 'response_hours', 'aid_amount', 'respo

#### CUIDADO: 
Cuando llegue a la regresión lineal tendré que eliminar una columna por categoría para evitar la trampa de las variables dummy (multicolinealidad perfecta). En lugar de 24 columnas (1 por país), me quedaré con 23, y cuando todas sean 0 será que es el país 24.

In [18]:
df_fe.to_csv('desastres_y_respuestas_limpio_FE.csv', index=False)